INSTALACIÓN DE LAS LIBRERÍAS NECESARIAS PARA CORRER EL CUADERNO

In [ ]:
%pip install --upgrade pip
%pip install -r requirements.txt

IMPORTAMOS LAS LIBRERÍAS QUE VAMOS A UTILIZAR

In [ ]:
import pandas as pd
import numpy as npÇ
import seaborn as sns
import matplotlib.pyplot as plt

DEFINIMOS LA RUTA LOCAL AL ARCHIVO CON LOS DATOS

In [ ]:
path = "data/dataset_sintetico_clientes.xlsx"
df = pd.read_excel(path) # Carga el archivo Excel en un DataFrame de pandas, necesita el paquete openpyxl
df.head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_sintetico_clientes.xlsx'

Visualizamos la información relevante sobre el tipo de datos

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   cliente_id                1000 non-null   int64  
 1   frecuencia_compra         1000 non-null   int64  
 2   monto_total               1000 non-null   float64
 3   categoria_producto        1000 non-null   object 
 4   edad_cliente              1000 non-null   int64  
 5   region                    1000 non-null   object 
 6   dias_desde_ultima_compra  1000 non-null   int64  
 7   satisfaccion              1000 non-null   int64  
 8   descuento_aplicado        1000 non-null   float64
 9   productos_comprados       1000 non-null   int64  
dtypes: float64(2), int64(6), object(2)
memory usage: 78.3+ KB


Vemos el resumen estadístico de los datos

In [ ]:
statistic_summary = df.describe()

for col in df.columns:
    if df[col].dtype in [np.float64, np.int64]:
        statistic_summary.at['median', col] = df[col].median()
        statistic_summary.at['mode', col] = df[col].mode()[0]
        statistic_summary.at['variance', col] = df[col].var()

statistic_summary.head(20)

,cliente_id,frecuencia_compra,monto_total,edad_cliente,dias_desde_ultima_compra,satisfaccion,descuento_aplicado,productos_comprados
count,1000.000000,1000.000000,1.000000e+03,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,500.500000,25.379000,2.575267e+03,43.967000,186.571000,3.002000,24.410256,10.019000
std,288.819436,14.122138,1.398215e+03,14.822470,103.053253,1.403555,14.297270,5.438034
min,1.000000,1.000000,5.117574e+01,18.000000,0.000000,1.000000,0.103826,1.000000
25%,250.750000,13.000000,1.381837e+03,31.000000,97.750000,2.000000,12.003801,6.000000
50%,500.500000,26.000000,2.656676e+03,44.000000,191.000000,3.000000,24.369769,10.000000
75%,750.250000,37.000000,3.753561e+03,57.000000,272.000000,4.000000,36.690982,15.000000
max,1000.000000,49.000000,4.996800e+03,69.000000,364.000000,5.000000,49.952475,19.000000
median,500.500000,26.000000,2.656676e+03,44.000000,191.000000,3.000000,24.369769,10.000000
mode,1.000000,26.000000,5.117574e+01,58.000000,238.000000,3.000000,0.103826,7.000000


Feature engineering

In [ ]:
df_clean = df.copy()

df_clean["grado_frecuencia"] = pd.qcut(
    df_clean["frecuencia_compra"],
    q=4,
    labels=["ocasional", "medio ocasional", "medio frecuente", "frecuente"]
)

df_clean["tipo_gastador"] = pd.qcut(
    df_clean["monto_total"],
    q=4,
    labels=["muy bajo", "bajo", "medio", "alto"]
)

df_clean["cliente_reciente"] = pd.qcut(
    df_clean["dias_desde_ultima_compra"],
    q=4,
    labels=["muy reciente", "reciente", "medio reciente", "poco reciente"]  
)

# pd.qcut divide los datos en cuantiles iguales basados en el valor de "frecuencia_compra"
# argumentos: columna del df, q=4 (cuartiles), labels (etiquetas para cada cuartil)

df_clean.head(10)

,cliente_id,frecuencia_compra,monto_total,categoria_producto,edad_cliente,region,dias_desde_ultima_compra,satisfaccion,descuento_aplicado,productos_comprados,grado_frecuencia,tipo_gastador,cliente_reciente
0,1,39,3499.959239,C,18,Sur,5,4,41.491063,3,frecuente,medio,muy reciente
1,2,29,4986.414897,D,25,Norte,203,2,32.736267,13,medio frecuente,alto,medio reciente
2,3,15,4488.220802,B,40,Oeste,238,2,20.560203,3,medio ocasional,alto,medio reciente
3,4,43,2901.192168,A,20,Este,184,5,33.168127,6,frecuente,medio,reciente
4,5,8,4591.108285,C,21,Norte,359,4,5.911121,11,ocasional,alto,poco reciente
5,6,21,76.235032,A,42,Norte,291,3,20.074881,17,medio ocasional,muy bajo,poco reciente
6,7,39,4876.582457,C,64,Sur,334,5,39.656427,10,frecuente,alto,poco reciente
7,8,19,2479.206457,B,55,Este,219,2,11.025439,12,medio ocasional,bajo,medio reciente
8,9,23,3628.335678,C,32,Este,115,4,9.298075,17,medio ocasional,medio,reciente
9,10,11,4113.264292,B,38,Este,312,5,47.540933,4,ocasional,alto,poco reciente


GRÁFICOS RELEVANTES

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Histograma de la columna "edad_cliente"
fig, axes = plt.subplots(2,1, figsize=(8, 8), sharex=True)

sns.histplot(data = df_clean, x='edad_cliente', bins=15 kde=True, edgecolor='white', ax=axes[0])   
for p in axes[0].patches:
    h = p.get_height()
    if h > 0:
        axes[0].annotate(f'{int(h)}',
                    (p.get_x() + p.get_width() / 2, h), 
                    ha='center', va='bottom',
                    fontsize=8, color='black')
axes[0].set_title('Distribución de la Edad de los Clientes')
axes[0].set_ylabel('Frecuencia')

# compras por edad
sns.histplot(data = df_clean, x='edad_cliente', weights='productos_comprados', kde=True, edgecolor='white', ax=axes[1])
for p in axes[1].patches:
    h = p.get_height()
    if h > 0:
        axes[1].annotate(f'{int(h)}',
                    (p.get_x() + p.get_width() / 2, h), 
                    ha='center', va='bottom',
                    fontsize=6, color='black')
axes[1].set_title('Distribución de Productos Comprados por Edad de los Clientes')
axes[1].set_xlabel('Edad')
axes[1].set_xlim(10, 80)
plt.show()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (666131512.py, line 4)

In [ ]:
# Satisfacción del cliente por tipo de cliente y gasto total
fig, axes = plt.subplots(2, 1, figsize=(8,8), sharex=True)
sns.barplot(data=df_clean, x='grado_frecuencia', y='satisfaccion', errorbar = None, ax=axes[0])
axes[0].set_title("Nivel de satisfacción por tipo de cliente")
axes[0].set_ylim(2.8,3.2)
for p in axes[0].patches:
    h = p.get_height()
    axes[0].annotate(f'{h:.2f}',
                (p.get_x() + p.get_width() / 2, h), 
                ha='center', va='bottom',
                fontsize=8, color='black')

sns.barplot(data=df_clean, x='grado_frecuencia', y='monto_total', errorbar = None, ax=axes[1])
for p in axes[1].patches:
    h = p.get_height()
    axes[1].annotate(f'{h:.2f}',
                (p.get_x() + p.get_width() / 2, h), 
                ha='center', va='bottom',
                fontsize=8, color='black')
axes[1].set_title("Monto total gastado por tipo de cliente")
axes[1].set_ylim(2300, 2800)